# Seasonal Agriculture Performance Analysis

**VOIS AICTE Major Project**

This notebook analyzes agricultural performance across **Kharif, Rabi and Zaid** seasons. It covers data understanding, cleaning, exploratory analysis, seasonal/crop comparisons, irrigation/resource analysis, correlation analysis, statistical testing, findings and recommendations.

> The analysis is based on the supplied Seasonal Agriculture Performance dataset. Results are presented as associations observed in the data; they should not be interpreted as causal effects without further study.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 5)

## 2. Load Dataset

In [ ]:
file_path = "seasonal_agriculture_performance_dataset.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Dataset Overview

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.info())
display(df.describe(include='all').T)

## 4. Missing Values and Duplicates

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0])

print("Duplicate rows:", df.duplicated().sum())

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()

# Seasonal median imputation for rainfall and soil moisture
df_clean['Rainfall_mm'] = df_clean.groupby('Season')['Rainfall_mm'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['Soil_Moisture_pct'] = df_clean.groupby('Season')['Soil_Moisture_pct'].transform(
    lambda x: x.fillna(x.median())
)

# Crop + season median imputation for yield
df_clean['Yield_Tonnes_Ha'] = df_clean.groupby(['Crop', 'Season'])['Yield_Tonnes_Ha'].transform(
    lambda x: x.fillna(x.median())
)

print("Remaining missing values:", int(df_clean.isnull().sum().sum()))
print("Remaining duplicate rows:", int(df_clean.duplicated().sum()))

## 6. Seasonal Distribution

In [ ]:
season_counts = df_clean['Season'].value_counts()
display(season_counts)

season_order = ['Kharif', 'Rabi', 'Zaid']
season_order = [s for s in season_order if s in df_clean['Season'].unique()]

plt.figure(figsize=(7,4))
season_counts.reindex(season_order).plot(kind='bar')
plt.title('Number of Records by Season')
plt.xlabel('Season')
plt.ylabel('Number of Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Seasonal Performance

In [ ]:
season_summary = df_clean.groupby('Season').agg(
    Average_Yield=('Yield_Tonnes_Ha','mean'),
    Average_Production=('Production_Tonnes','mean'),
    Average_Revenue=('Revenue_INR','mean'),
    Average_Cost=('Total_Cost_INR','mean'),
    Average_Profit=('Profit_INR','mean'),
    Average_Water_Used=('Water_Used_m3','mean'),
    Average_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean'),
    Average_Rainfall=('Rainfall_mm','mean')
).reindex(season_order)

display(season_summary.round(2))

### Average Yield by Season

In [ ]:
plt.figure(figsize=(8,5))
season_summary['Average_Yield'].plot(kind='bar')
plt.title('Average Yield by Season')
plt.xlabel('Season')
plt.ylabel('Yield (Tonnes/Ha)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Average Profit by Season

In [ ]:
plt.figure(figsize=(8,5))
season_summary['Average_Profit'].plot(kind='bar')
plt.title('Average Profit by Season')
plt.xlabel('Season')
plt.ylabel('Average Profit (INR)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Crop-wise and Season-wise Yield

In [ ]:
crop_season_yield = df_clean.pivot_table(
    values='Yield_Tonnes_Ha',
    index='Crop',
    columns='Season',
    aggfunc='mean'
)
crop_season_yield = crop_season_yield[
    [s for s in season_order if s in crop_season_yield.columns]
]
display(crop_season_yield.round(2))

crop_season_yield.plot(kind='bar', figsize=(11,6))
plt.title('Average Yield by Crop and Season')
plt.xlabel('Crop')
plt.ylabel('Yield (Tonnes/Ha)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Irrigation Method Analysis

In [ ]:
irrigation_summary = df_clean.groupby('Irrigation_Method').agg(
    Average_Yield=('Yield_Tonnes_Ha','mean'),
    Average_Water_Used=('Water_Used_m3','mean'),
    Average_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean'),
    Average_Profit=('Profit_INR','mean'),
    Records=('Season','size')
).sort_values('Average_Yield', ascending=False)

display(irrigation_summary.round(2))

irrigation_summary['Average_Yield'].plot(kind='bar', figsize=(8,5))
plt.title('Average Yield by Irrigation Method')
plt.xlabel('Irrigation Method')
plt.ylabel('Yield (Tonnes/Ha)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Correlation Analysis

In [ ]:
numeric_columns = df_clean.select_dtypes(include=np.number).columns
correlation = df_clean[numeric_columns].corr()

yield_corr = correlation['Yield_Tonnes_Ha'].sort_values(ascending=False)
display(yield_corr)

plt.figure(figsize=(12,9))
plt.imshow(correlation, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=90, fontsize=7)
plt.yticks(range(len(correlation.index)), correlation.index, fontsize=7)
plt.title('Correlation Matrix of Numeric Agricultural Variables')
plt.tight_layout()
plt.show()

## 11. Statistical Testing — One-Way ANOVA

In [ ]:
def anova_by_season(column):
    groups = [
        df_clean.loc[df_clean['Season'] == s, column].dropna()
        for s in season_order
    ]
    return f_oneway(*groups)

for column in ['Yield_Tonnes_Ha', 'Profit_INR', 'Water_Efficiency_t_per_1000m3']:
    f_stat, p_value = anova_by_season(column)
    print(f"{column}: F = {f_stat:.3f}, p = {p_value:.6g}")

## 12. Key Findings

1. Agricultural performance varies across Kharif, Rabi and Zaid.
2. Kharif records the highest average yield, revenue and profit in the analyzed data.
3. Zaid records the weakest average profitability and water efficiency.
4. Crop-level comparisons show a consistent seasonal pattern that should be interpreted alongside crop characteristics.
5. Irrigation methods show different average yield, water-use efficiency and profitability profiles.
6. Correlation analysis identifies which numeric variables are most strongly associated with yield, but correlation does not establish causation.
7. ANOVA can be used to test whether seasonal differences are statistically significant for selected performance measures.

## 13. Recommendations

- Use seasonal performance analysis to support agricultural planning.
- Consider water-use efficiency when evaluating irrigation practices, not only total yield.
- Evaluate crop and season combinations rather than comparing seasons in isolation.
- Investigate economically weak crop-season combinations using cost and market-price data.
- For future work, use regression or multivariate models to control for crop, region, farm size and other confounding factors before making causal claims.

## 14. Conclusion

The analysis demonstrates that agricultural performance is multidimensional. Seasonal, crop, environmental, resource and economic variables should be considered together. The results provide evidence of meaningful differences and relationships in the supplied dataset and can support more informed seasonal agricultural planning.